In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd
import optuna
from rdkit import Chem
from rdkit.Chem import Descriptors, MACCSkeys, AllChem, rdMolDescriptors
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

# 데이터 로드
train = pd.read_csv("./data/train.csv")
test = pd.read_csv("./data/test.csv")
submission = pd.read_csv("./data/sample_submission.csv")

# RDKit 특성 추출
def get_molecule_descriptors(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return [0] * 2232
        basic = [
            Descriptors.MolMR(mol),
            Descriptors.NumValenceElectrons(mol),
            Descriptors.MaxPartialCharge(mol),
            Descriptors.MinPartialCharge(mol),
            Descriptors.HeavyAtomMolWt(mol),
            Descriptors.ExactMolWt(mol),
            Descriptors.FpDensityMorgan1(mol),
            Descriptors.FpDensityMorgan2(mol),
            Descriptors.FpDensityMorgan3(mol),
            rdMolDescriptors.CalcNumBridgeheadAtoms(mol),
            rdMolDescriptors.CalcNumSpiroAtoms(mol),
            rdMolDescriptors.CalcChi0n(mol),
            rdMolDescriptors.CalcKappa1(mol)
        ]
        morgan_fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
        maccs_fp = MACCSkeys.GenMACCSKeys(mol)
        return basic + list(morgan_fp) + list(maccs_fp)
    except:
        return [0] * 2232

print("RDKit feature 생성 중...")
train["features"] = train["Canonical_Smiles"].apply(get_molecule_descriptors)
X_train = np.array(train["features"].tolist())
y_train = train["Inhibition"].values

test["features"] = test["Canonical_Smiles"].apply(get_molecule_descriptors)
X_test = np.array(test["features"].tolist())

# 스케일링
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# LightGBM Optuna 튜닝 (Normalized RMSE 최소화)
def objective(trial):
    params = {
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "device": "cpu",
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "lambda_l1": trial.suggest_float("lambda_l1", 0, 2.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0, 2.0)
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for train_idx, valid_idx in kf.split(X_train_scaled):
        X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[valid_idx]
        y_tr, y_val = y_train[train_idx], y_train[valid_idx]

        train_set = lgb.Dataset(X_tr, label=y_tr)
        valid_set = lgb.Dataset(X_val, label=y_val)

        model = lgb.train(
            params,
            train_set,
            valid_sets=[valid_set],
            valid_names=["valid"],
            callbacks=[lgb.early_stopping(stopping_rounds=20)],
        )
        preds = model.predict(X_val, num_iteration=model.best_iteration)
        rmse = mean_squared_error(y_val, preds, squared=False)
        scores.append(rmse)

    return np.mean(scores)

print("Optuna 탐색 중...")
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=100, show_progress_bar=True)
print("최적 파라미터:", study.best_params)

# 최적 LightGBM 재학습
print("LightGBM 최종 학습 중...")
best_params = study.best_params
best_params.update({
    "objective": "regression",
    "metric": "rmse",
    "verbosity": -1,
    "boosting_type": "gbdt",
    "device": "cpu"
})
train_set = lgb.Dataset(X_train_scaled, label=y_train)

model = lgb.train(
    best_params,
    train_set,
    num_boost_round=300
)
lgb_preds = model.predict(X_test_scaled)

# ChemProp 학습용 파일 저장
train_fp = "chemprop_train.csv"
test_fp = "chemprop_test.csv"
pred_fp = "chemprop_preds.csv"
model_dir = "chemprop_model"
model_path = "chemprop_model/model_0/best.pt"

train[["Canonical_Smiles", "Inhibition"]].to_csv(train_fp, index=False, header=["smiles", "target"])
test[["Canonical_Smiles"]].to_csv(test_fp, index=False, header=["smiles"])

# ChemProp 학습
print("ChemProp 학습 중 (CLI)...")
subprocess.run([
    "python", "-m", "chemprop.cli.train",
    "--data_path", train_fp,
    "--dataset_type", "regression",
    "--save_dir", model_dir,
    "--target_columns", "target",
    "--epochs", "50",
    "--hidden_size", "500",
    "--depth", "5",
    "--dropout", "0.3",
    "--ensemble_size", "3",
    "--accelerator", "gpu",
    "--devices", "1"
], capture_output=True, text=True)

# ChemProp 예측
if not os.path.exists(model_path):
    raise FileNotFoundError(f"ChemProp 모델이 없습니다: {model_path}")

print("ChemProp 예측 중 (CLI)...")
subprocess.run([
    "python", "-m", "chemprop.cli.predict",
    "--test_path", test_fp,
    "--model_paths", model_path,
    "--preds_path", pred_fp
], capture_output=True, text=True)

# 앙상블 가중치 튜닝
print("앙상블 가중치 튜닝 중...")
chemprop_preds = pd.read_csv(pred_fp)
chemprop_preds = chemprop_preds.iloc[:, 1].values.astype(float)

def ensemble_objective(trial):
    w = trial.suggest_float("weight", 0.0, 1.0)
    ensemble = w * lgb_preds + (1 - w) * chemprop_preds
    return np.std(ensemble) 

w_study = optuna.create_study(direction="minimize")
w_study.optimize(ensemble_objective, n_trials=30)
best_weight = w_study.best_params["weight"]
ensemble_preds = best_weight * lgb_preds + (1 - best_weight) * chemprop_preds
print(f"최적 앙상블 비율: LGB {best_weight:.3f} / ChemProp {1 - best_weight:.3f}")

# 제출
submission["Inhibition"] = ensemble_preds
os.makedirs("sub", exist_ok=True)
submission.to_csv("./sub/submission.csv", index=False)
print("완료")

📌 RDKit feature 생성 중...


[18:36:14] DEPRECATION WARNING: please use MorganGenerator
[18:36:14] DEPRECATION WARNING: please use MorganGenerator
[18:36:14] DEPRECATION WARNING: please use MorganGenerator
[18:36:14] DEPRECATION WARNING: please use MorganGenerator
[18:36:14] DEPRECATION WARNING: please use MorganGenerator
[18:36:14] DEPRECATION WARNING: please use MorganGenerator
[18:36:14] DEPRECATION WARNING: please use MorganGenerator
[18:36:14] DEPRECATION WARNING: please use MorganGenerator
[18:36:14] DEPRECATION WARNING: please use MorganGenerator
[18:36:14] DEPRECATION WARNING: please use MorganGenerator
[18:36:14] DEPRECATION WARNING: please use MorganGenerator
[18:36:14] DEPRECATION WARNING: please use MorganGenerator
[18:36:14] DEPRECATION WARNING: please use MorganGenerator
[18:36:14] DEPRECATION WARNING: please use MorganGenerator
[18:36:14] DEPRECATION WARNING: please use MorganGenerator
[18:36:14] DEPRECATION WARNING: please use MorganGenerator
[18:36:14] DEPRECATION WARNING: please use MorganGenerat

🔍 Optuna 탐색 중...


  0%|          | 0/100 [00:00<?, ?it/s]

Training until validation scores don't improve for 20 rounds
Did not meet early stopping. Best iteration is:
[100]	valid's rmse: 24.0114


...

Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[53]	valid's rmse: 24.6966
Training until validation scores don't improve for 20 rounds
Did not meet early stopping. Best iteration is:
[100]	valid's rmse: 23.289


c:\Users\user\anaconda3\envs\da\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
c:\Users\user\anaconda3\envs\da\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[35]	valid's rmse: 24.2168
[I 2025-07-08 18:37:14,550] Trial 99 finished with value: 23.722421768605457 and parameters: {'learning_rate': 0.08520825866238067, 'num_leaves': 28, 'max_depth': 4, 'feature_fraction': 0.9559127229128047, 'bagging_fraction': 0.7853859517445655, 'bagging_freq': 9, 'lambda_l1': 0.2661411624591915, 'lambda_l2': 1.046125896112681}. Best is trial 18 with value: 23.566374559434276.
✅ 최적 파라미터: {'learning_rate': 0.06337706259457217, 'num_leaves': 47, 'max_depth': 11, 'feature_fraction': 0.7005864154724466, 'bagging_fraction': 0.7400395445146233, 'bagging_freq': 6, 'lambda_l1': 0.4472838095907976, 'lambda_l2': 1.6511321733043922}
⚙️ LightGBM 최종 학습 중...
🚀 ChemProp 학습 중 (CLI)...
🔮 ChemProp 예측 중 (CLI)...


[I 2025-07-08 18:37:28,804] A new study created in memory with name: no-name-cc92a90d-85a6-4504-bc1f-bf7fa60be1c5
[I 2025-07-08 18:37:28,805] Trial 0 finished with value: 12.34566784578496 and parameters: {'weight': 0.8322744313944966}. Best is trial 0 with value: 12.34566784578496.
[I 2025-07-08 18:37:28,805] Trial 1 finished with value: 13.077354516908835 and parameters: {'weight': 0.0850941372242161}. Best is trial 0 with value: 12.34566784578496.
[I 2025-07-08 18:37:28,806] Trial 2 finished with value: 12.219247232724086 and parameters: {'weight': 0.4166918902597674}. Best is trial 2 with value: 12.219247232724086.
[I 2025-07-08 18:37:28,806] Trial 3 finished with value: 12.99708610344491 and parameters: {'weight': 0.10696498463715953}. Best is trial 2 with value: 12.219247232724086.
[I 2025-07-08 18:37:28,806] Trial 4 finished with value: 12.135227918667232 and parameters: {'weight': 0.6743683017224459}. Best is trial 4 with value: 12.135227918667232.
[I 2025-07-08 18:37:28,807] T

🎯 앙상블 가중치 튜닝 중...
✅ 최적 앙상블 비율: LGB 0.589 / ChemProp 0.411
✅ 
